In [51]:
import pandas as pd

In [52]:
df = pd.read_csv("../data/ote_hourly_interval_prices.csv", sep=";") #select correct path (there is no such file in this repo, use your own)

In [53]:
df["Max_Hodina"] = df.groupby("Den")["Hodina"].transform("max")
df_15min = df.loc[df.index.repeat(4)].copy()

In [54]:
df_15min["Quarter"] = df_15min.groupby(level=0).cumcount()
df_15min["Perioda"] = (df_15min["Hodina"] - 1) * 4 + df_15min["Quarter"] + 1

In [55]:
def get_actual_clock_time(row):
    h = int(row["Hodina"])
    max_h = int(row["Max_Hodina"])
    q = int(row["Quarter"])
    
    # Spring DST (23-hour day): clock jumps from 02:00 to 03:00
    if max_h == 23:
        if h >= 3:
            start_hour = h
        else:
            start_hour = h - 1
            
    # Autumn DST (25-hour day): 02:00 to 03:00 happens twice
    elif max_h == 25:
        if h >= 5:
            start_hour = h - 2
        elif h == 4:
            start_hour = 2
        else:
            start_hour = h - 1
            
    # Standard 24-hour day
    else:
        start_hour = h - 1
        
    start_min = q * 15
    end_min = (q + 1) * 15
    end_hour = start_hour
    
    if end_min == 60:
        end_min = 0
        end_hour += 1
        
    return f"{start_hour:02d}:{start_min:02d}-{end_hour:02d}:{end_min:02d}"

In [56]:
df_15min["Časový interval"] = df_15min.apply(get_actual_clock_time, axis=1)

In [57]:
df_15min = df_15min.drop(columns=["Hodina", "Quarter", "Max_Hodina"])

In [58]:
cols = ["Den", "Perioda", "Časový interval", "Nákup (MWh)", "Prodej (MWh)", 
        "Saldo DT\n(MWh)", "Export\n(MWh)", "Import\n(MWh)", 
        "Množství - vč. Exp a Imp (MWh)", "Marginální cena ČR (EUR/MWh)", 
        "Marginální cena ČR (Kč/MWh)", "Kurz Kč/EUR (ČNB)", "Částka (EUR)"]

# Handle any subtle naming discrepancies in columns (like newline characters)
existing_cols = [c for c in cols if c in df_15min.columns]
df_15min = df_15min[existing_cols]

In [59]:
df_15min.to_csv("../data/ote_test_15min.csv", sep=";", index=False)